# ASLytics: gloss pairs and gloss model on Kaggle

Generates English → ASL gloss pairs with an open model, then trains the gloss translator, all on Kaggle's free GPU.

**Before running**
1. On your Mac: `python prepare_kaggle_inputs.py --how2sign ../../data/how2sign_text/how2sign_realigned_train.csv`
2. Upload the `kaggle_inputs` folder as a **private** Kaggle dataset (it derives from How2Sign, which is non-commercial) and attach it to this notebook.
3. Notebook settings: Accelerator **GPU T4 x2**, Internet **On**.

Generation is resumable. If the session ends, start it again and re-run: finished sentences are skipped as long as `/kaggle/working/pairs_kaggle.jsonl` is still there (use "Save Version" to keep it between sessions, or download it).

In [ ]:
import glob, os, shutil
found = glob.glob('/kaggle/input/**/glosses.json', recursive=True)
assert found, 'Attach the kaggle_inputs dataset to this notebook first.'
INPUTS = os.path.dirname(found[0])
WORK = '/kaggle/working'
for script in ('kaggle_generate.py', 'train.py'):
    shutil.copy(os.path.join(INPUTS, script), WORK)
os.chdir(WORK)
# Use the newest scripts from GitHub when reachable, so fixes do not need a dataset re-upload.
RAW = 'https://raw.githubusercontent.com/Japleen-Kaur2409/ASLytics-new/prototype/training/text2gloss/'
for script in ('kaggle_generate.py', 'train.py'):
    os.system(f'wget -q -O {script}.new {RAW}{script} && mv {script}.new {script}')
print('inputs:', INPUTS, os.listdir(INPUTS))
!pip -q install -U bitsandbytes accelerate sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Trial: 40 sentences

Read the output before running the full set. If most lines are dropped or the glosses look like English word order, try a different `MODEL`.

In [ ]:
MODEL = 'Qwen/Qwen2.5-7B-Instruct'   # any chat model on Hugging Face that fits a 16 GB GPU in 4-bit
!rm -f trial.jsonl
!python kaggle_generate.py --inputs "$INPUTS" --out trial.jsonl --model "$MODEL" --limit 40 2>&1 | tail -3
import json, collections
rows = [json.loads(l) for l in open('trial.jsonl')]
print(sum(r['ok'] for r in rows), 'of', len(rows), 'kept')
print('words that were not signs:', collections.Counter(t for r in rows for t in r['invalid']).most_common(25), '\n')
for r in rows[:25]:
    print(r['english'])
    print('   ->', r['gloss'] if r['ok'] else f"(dropped) model wrote: {r['raw']}")

## 2. Full run

About 15,000 sentences. This can take several hours; lower `--limit` for a first pass.

In [ ]:
!python kaggle_generate.py --inputs "$INPUTS" --out pairs_kaggle.jsonl --model "$MODEL" --limit 15000

## 3. Train the gloss model twice and compare

Both runs are scored on the same held-out Claude-written pairs. If adding the Kaggle pairs does not raise `eval_token_f1`, do not use them.

In [ ]:
!python train.py --pairs "$INPUTS/claude_pairs.jsonl" --out model_claude_only --epochs 15 --batch-size 32 --lr 1e-3 2>&1 | grep -E "eval_token_f1|expected|got|held out" | tail -20

In [ ]:
!python train.py --pairs "$INPUTS/claude_pairs.jsonl" --extra pairs_kaggle.jsonl --out model_with_kaggle --epochs 10 --batch-size 64 --lr 1e-3 2>&1 | grep -E "eval_token_f1|expected|got|held out" | tail -20

## 4. Download

Download `models.zip` and `pairs_kaggle.jsonl` from the Output panel. On your Mac, unzip into `training/text2gloss/` and run `python export_onnx.py --model <folder> --out onnx_model/` to make the browser version (the export is done locally because it needs a specific PyTorch version).

In [ ]:
!rm -rf model_claude_only/checkpoints model_with_kaggle/checkpoints
!zip -qr models.zip model_claude_only model_with_kaggle
!ls -la models.zip pairs_kaggle.jsonl